# LAB: Agents that Speak MCP Fluently

**Ayhan Demir** · Ironhack AI Consulting & Integration Bootcamp · Module 4, Week 4

## Objective

In this lab we connect **three MCP servers** that provide different kinds of information:

- 🌐 **Fetch** runs on our laptop and reads public web pages.
- 🐙 **GitHub** runs remotely and reads repository facts through a read-only connection.
- 📚 **LangChain Docs** runs remotely and provides a documentation resource.

The goal is a **provably grounded** agent: for every answer, we can identify the tool call or resource text that supports it.

## Use case: agent framework shortlist for a client RFP

A client asks which agent framework to shortlist. A consultant needs **facts, not opinions**: each framework's license and latest release from **GitHub**, what each framework says it is for from its **documentation page** read with **Fetch**, and LangGraph selection guidance from the **LangChain Docs MCP resource**. We compare **LangGraph** and the **OpenAI Agents SDK**.

## 🗺️ Steps we will follow

| # | Step | What happens |
|---|------|--------------|
| 1 | **Setup and installation** | Install Python packages, load the OpenAI key and GitHub token, and check uvx; LangChain Docs needs no token |
| 2 | **Connect to MCP servers** | Connect Fetch (stdio), GitHub (HTTP + token), and LangChain Docs (HTTP); list their tools and resources |
| 3 | **Load MCP tools into LangChain** | Load Fetch and GitHub tools; keep the Docs resource as read-only context |
| 4 | **Create agents with MCP tools** | Ask one question per tool server and inspect the tool calls behind each answer |
| 5 | **Access an MCP resource** | Give an agent read-only background from the LangChain Docs server |
| 6 | **Build a complete MCP-enabled agent** | Use three MCP clients, Fetch and GitHub tools, and the Docs resource in one RFP answer |
| 7 | Optional: MCP vs direct API | Not done (optional) |
| 8 | Optional: practical example | Not done (optional) |
| ✅ | **Save lab proof** | Update lab_proof.md with this three-server run after executing the notebook |

**Sources**
- Documentation pages: https://docs.langchain.com/oss/python/langgraph/overview and https://openai.github.io/openai-agents-python/
- GitHub repositories: langchain-ai/langgraph and openai/openai-agents-python
- LangChain Docs MCP endpoint and resource: https://docs.langchain.com/mcp

> If uvx is missing after installation, restart the kernel. Never commit .env or share its values.


---
## Step 1: Setup and Installation

Keep a .env file next to this notebook and make sure it is git-ignored. **Never commit it or print its values.**

| Service | Add to .env |
|---|---|
| All | OPENAI_API_KEY=sk-... |
| Fetch | Nothing |
| LangChain Docs | Nothing; the remote MCP endpoint needs no token |
| **GitHub** | GITHUB_PERSONAL_ACCESS_TOKEN=github_pat_... (fine-grained, *Public repositories (read-only)*) |

The notebook installs uv and the Python packages it uses. Fetch runs locally through uvx. If uvx is still missing after installation, restart the kernel. LangChain Docs is a remote MCP server, so it needs no local installation.


In [12]:
%pip install -qU uv langchain langchain-openai langchain-mcp-adapters langgraph mcp python-dotenv "pydantic>=2,<3"

Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
import sys
import shutil
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Add it to your git-ignored .env file.")

GITHUB_TOKEN = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")   # only needed for GitHub
# uvx starts the Fetch server: from Homebrew, or from this environment (pip install uv)
UVX = shutil.which("uvx") or os.path.join(os.path.dirname(sys.executable), "uvx")

print("OpenAI key:", "loaded" if os.getenv("OPENAI_API_KEY") else "missing")
print("uvx:", "found" if os.path.exists(UVX) else "NOT found: install uv, then restart the kernel")
print("GitHub token:", "loaded" if GITHUB_TOKEN else "missing: needed for GitHub")
print("LangChain Docs: remote MCP endpoint; no token required")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


OpenAI key: loaded
uvx: found
GitHub token: loaded
LangChain Docs: remote MCP endpoint; no token required


In [14]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

async def log_tool_calls(request, handler):
    # BEFORE: which server, which tool, which inputs
    print(f"[LOG] {request.server_name} -> {request.name}  {request.args}")
    result = await handler(request)   # the real tool runs on the server
    # AFTER: what the server sent back
    text = " ".join(getattr(part, "text", str(part)) for part in result.content)
    print(f"[LOG]    returned: {text[:150]}...")
    return result

**Done when:** the OpenAI key is loaded, uvx is found, the GitHub token is loaded, and you know LangChain Docs needs no credential. The cell reports status only; it never prints secret values.


---
## Step 2: Connect to the Three MCP Servers

**Goal:** create one client per MCP server, then check each client's tools and resources.

| | 🌐 Fetch | 🐙 GitHub | 📚 LangChain Docs |
|---|---|---|---|
| **Transport** | stdio: the client starts the server | http: the client calls the server | http: the client calls the server |
| **Connection** | command + args | url + headers | url |
| **Secret** | None | Read-only GitHub token in headers | None |
| **Used for** | Read public documentation pages | Read repository licenses and releases | Read-only LangChain guidance resource |

### 2a. Fetch client (stdio)

The client starts the pinned Python server as if we typed this in a terminal:

    uvx mcp-server-fetch@2026.10.24


In [15]:
# Fetch client: local server, started by uvx
fetch_client = MultiServerMCPClient(
    {
        "fetch": {
            "transport": "stdio",
            "command": UVX,
            "args": ["mcp-server-fetch@2026.10.24"],
        }
    },
    tool_interceptors=[log_tool_calls],
)
print("Fetch client configured")

Fetch client configured


### 2b. GitHub client (HTTP + token)

GitHub hosts its official MCP server at https://api.githubcopilot.com/mcp/. We send headers with every request:

| Header | Value | Why |
|---|---|---|
| Authorization | f"Bearer {GITHUB_TOKEN}" | Authenticates the read-only request |
| X-MCP-Toolsets | repos,issues | Loads only these groups of tools |
| X-MCP-Readonly | true | Restricts the server to read tools |

The npm package @modelcontextprotocol/server-github is deprecated, so this lab uses GitHub's official server.


In [16]:
# GitHub client: remote server, token in the headers
github_client = MultiServerMCPClient(
    {
        "github": {
            "transport": "http",
            "url": "https://api.githubcopilot.com/mcp/",
            "headers": {
                "Authorization": f"Bearer {GITHUB_TOKEN}",
                "X-MCP-Toolsets": "repos,issues",
                "X-MCP-Readonly": "true",
            },
        }
    },
    tool_interceptors=[log_tool_calls],
)
print("GitHub client configured")

GitHub client configured


### 2c. LangChain Docs client (HTTP, no token)

LangChain Docs exposes an MCP endpoint at https://docs.langchain.com/mcp. This is a remote MCP server, separate from fetching ordinary web pages. We connect to it here and use its resource in Step 5.


In [17]:
# LangChain Docs client: remote MCP server, no token
docs_client = MultiServerMCPClient(
    {"langchain-docs": {"transport": "http", "url": "https://docs.langchain.com/mcp"}},
    tool_interceptors=[log_tool_calls],
)
print("LangChain Docs client configured")


LangChain Docs client configured


### 2d. Test all three connections

For each client, count its tools and try to list its resources. Fetch may take about 20 seconds the first time while uvx downloads the pinned server.


In [18]:
clients = {
    "fetch": fetch_client,
    "github": github_client,
    "langchain-docs": docs_client,
}

for name, client in clients.items():
    tools = await client.get_tools()
    print(f"{name}: {len(tools)} tools")
    try:
        resources = await client.get_resources()
        print(f"{name}: {len(resources)} resources")
    except Exception as error:
        print(f"{name}: resource listing unavailable ({type(error).__name__})")


fetch: 1 tools
fetch: 0 resources
github: 19 tools
github: 1 resources
langchain-docs: 3 tools
langchain-docs: 1 resources


**Done when:** all three clients return a tool count and the resource listing shows which server provides read-only context. Fetch may have no resources; LangChain Docs is the resource source used later.


---
## Step 3: Load MCP Tools into LangChain

**Goal:** load the Fetch and GitHub tools that the agents will call. The LangChain Docs client is connected too, but this lab uses its resource as context rather than adding its tools to the agent.


In [19]:
fetch_tools = await fetch_client.get_tools()
github_tools = await github_client.get_tools()

print("Fetch tools: ", [t.name for t in fetch_tools])
print("GitHub tools:", [t.name for t in github_tools])


Fetch tools:  ['fetch']
GitHub tools: ['get_commit', 'get_file_contents', 'get_label', 'get_latest_release', 'get_release_by_tag', 'get_tag', 'issue_read', 'list_branches', 'list_commits', 'list_issue_fields', 'list_issue_types', 'list_issues', 'list_releases', 'list_repository_collaborators', 'list_tags', 'search_code', 'search_commits', 'search_issues', 'search_repositories']


**Done when:** the Fetch and GitHub tool names are visible. The LangChain Docs resource is retrieved separately in Step 5.
---
## Step 4: Create Agents with MCP Tools

**Goal:** create one agent per tool server, Fetch and GitHub, and inspect the calls behind each answer. Each question asks for information the model should retrieve from its source.


In [20]:
# Fetch agent: answers from a documentation page
fetch_agent = create_agent(
    model=llm,
    tools=fetch_tools,
    system_prompt=(
        "You read web pages with the fetch tool. Always fetch the page before you answer, "
        "never guess, and name the URL you used."
    ),
)

question = "Read https://openai.github.io/openai-agents-python/. In one or two sentences: what does the OpenAI Agents SDK say it is for?"

fetch_result = await fetch_agent.ainvoke({"messages": [HumanMessage(content=question)]})
print("\nAnswer:", fetch_result["messages"][-1].content)

[LOG] fetch -> fetch  {'url': 'https://openai.github.io/openai-agents-python/'}
[LOG]    returned: Contents of https://openai.github.io/openai-agents-python/:
<p>OpenAI Agents SDK</p>

Skip to content

# OpenAI Agents SDK

The OpenAI Agents SDK enab...
[LOG] fetch -> fetch  {'url': 'https://openai.github.io/openai-agents-python/', 'start_index': 5000}
[LOG]    returned: Contents of https://openai.github.io/openai-agents-python/:
esume state | Results |
| Build a low-latency voice agent with gpt-realtime-2.1 | Realtime...

Answer: The OpenAI Agents SDK is designed to enable the creation of agentic AI applications in a lightweight and user-friendly manner, providing essential tools for building agents that can perform tasks, delegate work, and ensure input/output validation. It aims to simplify the development process while allowing for complex interactions and real-world applications. 

You can find more information at [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/).

In [21]:
# GitHub agent: answers from a repository
github_agent = create_agent(
    model=llm,
    tools=github_tools,
    system_prompt=(
        "You read public GitHub repositories with the GitHub tools. Never guess: every fact "
        "must come from a tool result. Name the repository and the file or tool you used."
    ),
)

question = (
    "For the GitHub repository langchain-ai/langgraph: which license does it use "
    "(read the LICENSE file), and what is its latest release?"
)

github_result = await github_agent.ainvoke({"messages": [HumanMessage(content=question)]})
print("\nAnswer:", github_result["messages"][-1].content)

[LOG] github -> get_file_contents  {'owner': 'langchain-ai', 'repo': 'langgraph', 'path': 'LICENSE'}
[LOG] github -> get_latest_release  {'owner': 'langchain-ai', 'repo': 'langgraph'}
[LOG]    returned: {"tag_name":"1.2.18","target_commitish":"3521f08e21a7d89c28331191374bc85c11ecb6e2","name":"langgraph==1.2.18","body":"Changes since 1.2.17\n\n* releas...
[LOG]    returned: successfully downloaded text file (SHA: fc0602feecdd6748623c852ab534e1ca612673c7) type='resource' resource=TextResourceContents(uri=AnyUrl('repo://lan...

Answer: The GitHub repository **langchain-ai/langgraph** uses the **MIT License**. The license text states that permission is granted to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the software, with the condition that the copyright notice and permission notice are included in all copies or substantial portions of the software.

The latest release of the repository is **version 1.2.18**, which was published on **October 26, 2026

---
## Step 5: Access an MCP Resource

**Goal:** give an agent read-only background knowledge instead of tools.

| | Tools | Resources |
|---|---|---|
| **What** | Actions: *do* something | Read-only data: *know* something |
| **Who decides** | The agent calls them | Our code fetches them and hands them over |

Step 2d checks the resource lists: Fetch offers no resources, while LangChain Docs provides the guide used here. We reuse the LangChain Docs client created in Step 2. The guide fits the RFP because it describes when to choose LangGraph.


In [22]:
# Reuse the LangChain Docs client created in Step 2
docs_resources = await docs_client.get_resources()
if not docs_resources:
    raise RuntimeError("The LangChain Docs MCP server returned no resources.")

guide_text = docs_resources[0].as_string()
print(f"Loaded {len(docs_resources)} LangChain Docs resource(s); guide has {len(guide_text)} characters")

# Agent with NO tools: the guide text is its only source
guide_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "Answer ONLY from the guide below. If the guide does not say it, answer 'Not in the guide'.\n\n"
        f"GUIDE:\n{guide_text}"
    ),
)

question = "According to the guide, when should a team use LangGraph instead of LangChain's create_agent?"

guide_result = await guide_agent.ainvoke({"messages": [HumanMessage(content=question)]})
answer = guide_result["messages"][-1].content
print("\nAnswer:", answer)

# Evidence: check guide phrases in both the answer and resource text
for word in ["deterministic", "human-in-the-loop"]:
    print(f"'{word}': in answer = {word in answer.lower()}, in guide = {word in guide_text.lower()}")


Loaded 1 LangChain Docs resource(s); guide has 11252 characters

Answer: A team should use LangGraph instead of LangChain's create_agent when they need low-level control for complex workflows that mix deterministic logic with LLM-driven steps. LangGraph is suitable for scenarios requiring human-in-the-loop processes and deterministic steps.
'deterministic': in answer = True, in guide = True
'human-in-the-loop': in answer = True, in guide = True


---
## Step 6: Build a Complete MCP-Enabled Agent

**Goal:** answer one client RFP question using all three MCP connections: Fetch and GitHub tools for source retrieval, plus the LangChain Docs resource as read-only selection guidance. Resources are fetched by our code and passed as context; they are not added to the tool list.


In [23]:
# Two tool clients -> one tool list; the third client's resource stays in context
all_tools = fetch_tools + github_tools

print(f"{len(fetch_tools)} Fetch tools + {len(github_tools)} GitHub tools = {len(all_tools)} tools")
print([t.name for t in all_tools])
print(f"LangChain Docs resource context: {len(guide_text)} characters")


1 Fetch tools + 19 GitHub tools = 20 tools
['fetch', 'get_commit', 'get_file_contents', 'get_label', 'get_latest_release', 'get_release_by_tag', 'get_tag', 'issue_read', 'list_branches', 'list_commits', 'list_issue_fields', 'list_issue_types', 'list_issues', 'list_releases', 'list_repository_collaborators', 'list_tags', 'search_code', 'search_commits', 'search_issues', 'search_repositories']
LangChain Docs resource context: 11252 characters


In [24]:
combined_agent = create_agent(
    model=llm,
    tools=all_tools,
    system_prompt=(
        "You help an AI consultant shortlist agent frameworks for a client RFP. "
        "Use GitHub for repository facts and only accept results for the exact owner/repo requested; never substitute a demo or lookalike repository. "
        "Read the LICENSE file and latest release for each named repository. "
        "Use Fetch to read the exact official documentation URLs in the question. "
        "Use the LangChain Docs resource below only for LangGraph selection guidance. "
        "Name the repository, file, URL, or resource behind each claim. "
        "If a source does not provide a fact, say 'not found in the sources'. "
        "Do not use the LangChain guide to make claims about the OpenAI Agents SDK.\n\n"
        f"LANGCHAIN DOCS MCP RESOURCE (selection guidance):\n{guide_text}"
    ),
)

question = (
    "We are shortlisting agent frameworks for a client RFP. "
    "For LangGraph, use only GitHub repo langchain-ai/langgraph and docs "
    "https://docs.langchain.com/oss/python/langgraph/overview. "
    "For the OpenAI Agents SDK, use only GitHub repo openai/openai-agents-python and docs "
    "https://openai.github.io/openai-agents-python/. "
    "Give each framework's license and latest release from GitHub, and one sentence from its docs page on what it is for. "
    "Then add a short selection note: according to the LangChain Docs MCP resource, what kind of workflow is a reason to choose LangGraph over LangChain's create_agent? "
    "Answer with a compact table for the two frameworks and a separate selection note; include a source for each claim."
)

combined_result = await combined_agent.ainvoke({"messages": [HumanMessage(content=question)]})
print("\nAnswer:\n", combined_result["messages"][-1].content)


[LOG] github -> get_latest_release  {'owner': 'langchain-ai', 'repo': 'langgraph'}
[LOG] github -> get_latest_release  {'owner': 'openai', 'repo': 'openai-agents-python'}
[LOG] github -> get_file_contents  {'owner': 'langchain-ai', 'repo': 'langgraph', 'path': 'LICENSE'}
[LOG] github -> get_file_contents  {'owner': 'openai', 'repo': 'openai-agents-python', 'path': 'LICENSE'}
[LOG] fetch -> fetch  {'url': 'https://docs.langchain.com/oss/python/langgraph/overview'}
[LOG] fetch -> fetch  {'url': 'https://openai.github.io/openai-agents-python/'}
[LOG]    returned: Contents of https://openai.github.io/openai-agents-python/:
<p>OpenAI Agents SDK</p>

Skip to content

# OpenAI Agents SDK

The OpenAI Agents SDK enab...
[LOG]    returned: Contents of https://docs.langchain.com/oss/python/langgraph/overview:
<p>LangGraph overview - Docs by LangChain</p>

> ## Documentation Index
>
> Fetc...
[LOG]    returned: {"tag_name":"v0.24.0","target_commitish":"17d994bd3045dae71acda676a4ef4762ed8ecd94","na

### One question of our own: a source-boundary probe

The named repositories, documentation pages, and LangChain resource may not contain every RFP fact. Ask about paid enterprise support and annual pricing for the exact repositories. A grounded answer should report when those sources do not provide the information.


In [25]:
question = (
    "For the exact repositories langchain-ai/langgraph and openai/openai-agents-python, "
    "do the repository contents, their official documentation pages, or the LangChain Docs resource state whether paid enterprise support is available and its annual price? "
    "Use only those sources. If they do not state it, say 'not found in the sources'."
)

own_result = await combined_agent.ainvoke({"messages": [HumanMessage(content=question)]})
print("\nAnswer:\n", own_result["messages"][-1].content)


[LOG] github -> get_file_contents  {'owner': 'langchain-ai', 'repo': 'langgraph', 'path': 'README.md'}
[LOG] github -> get_file_contents  {'owner': 'openai', 'repo': 'openai-agents-python', 'path': 'README.md'}
[LOG] fetch -> fetch  {'url': 'https://docs.langchain.com'}
[LOG] fetch -> fetch  {'url': 'https://github.com/openai/openai-agents-python'}
[LOG]    returned: Contents of https://docs.langchain.com/:
<p>Home - Docs by LangChain</p>

> ## Documentation Index
>
> Fetch the complete documentation index at: /llm...
[LOG]    returned: Contents of https://github.com/openai/openai-agents-python:
<p>GitHub - openai/openai-agents-python: A lightweight, powerful framework for multi-agent...
[LOG]    returned: successfully downloaded text file (SHA: 40f490c110cd584f7034bd221c6da37379fc41ff) type='resource' resource=TextResourceContents(uri=AnyUrl('repo://ope...
[LOG]    returned: successfully downloaded text file (SHA: 97c31e9cb4d8fe56be8d768ce3eb5e22400e897e) type='resource' resource=Text